In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType
from pyspark.sql.window import Window

storage_account = "energybigdatastorage"

# ÉTAPE 0 : LECTURE
df_hh = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .load(f"abfss://raw@{storage_account}.dfs.core.windows.net/energy_data_extracted/archive (3).zip/halfhourly_dataset/halfhourly_dataset")

# ÉTAPE 1 : DOUBLONS
df_hh = df_hh.dropDuplicates()

# ÉTAPE 2 : STANDARDISER + TRIM + UPPER LCLid
df_hh = df_hh.withColumn("energy(kWh/hh)",
    F.when(F.col("energy(kWh/hh)") == "Null", None)
    .otherwise(F.col("energy(kWh/hh)")))
df_hh = df_hh.withColumn("energy_kwh",
    F.col("energy(kWh/hh)").cast(FloatType()))
df_hh = df_hh.drop("energy(kWh/hh)")
df_hh = df_hh.withColumn("LCLid", F.upper(F.trim(F.col("LCLid"))))

# ÉTAPE 3 : NULLS FORWARD FILL
window_spec = Window.partitionBy("LCLid").orderBy("tstp")
df_hh = df_hh.withColumn("energy_kwh",
    F.last("energy_kwh", ignorenulls=True).over(window_spec))

# ÉTAPE 4 : FORMAT DATES
df_hh = df_hh.withColumn("tstp", F.to_timestamp("tstp"))

# ÉTAPE 5 : FLAGGING OUTLIERS
quantiles = df_hh.approxQuantile("energy_kwh", [0.25, 0.75], 0.05)
Q1, Q3 = quantiles[0], quantiles[1]
IQR = Q3 - Q1
df_hh = df_hh.withColumn("is_outlier",
    F.when(
        (F.col("energy_kwh") < Q1 - 1.5 * IQR) |
        (F.col("energy_kwh") > Q3 + 1.5 * IQR), 1)
    .otherwise(0))

# ÉTAPE 6 : FEATURES TEMPORELLES
df_hh = df_hh.withColumn("hour", F.hour("tstp"))
df_hh = df_hh.withColumn("day_of_week", F.dayofweek("tstp"))
df_hh = df_hh.withColumn("month", F.month("tstp"))
df_hh = df_hh.withColumn("year", F.year("tstp"))
df_hh = df_hh.withColumn("is_weekend",
    F.when(F.col("day_of_week").isin([1, 7]), 1).otherwise(0))
df_hh = df_hh.withColumn("quarter", F.quarter("tstp"))

# ÉTAPE 7 : CONSOMMATION MOYENNE PAR FOYER
df_avg = df_hh.groupBy("LCLid").agg(
    F.mean("energy_kwh").alias("avg_energy_kwh"),
    F.min("energy_kwh").alias("min_energy_kwh"),
    F.max("energy_kwh").alias("max_energy_kwh"),
    F.stddev("energy_kwh").alias("std_energy_kwh"),
    F.count("energy_kwh").alias("nb_mesures")
)

# ÉTAPE 8 : SAUVEGARDE
df_hh.write.format("delta").mode("overwrite") \
    .partitionBy("year", "month") \
    .save(f"abfss://processed@{storage_account}.dfs.core.windows.net/halfhourly_v2/")
print("✅ halfhourly_v2 sauvegardé !")

df_avg.write.format("delta").mode("overwrite") \
    .save(f"abfss://processed@{storage_account}.dfs.core.windows.net/halfhourly_avg_v2/")
print("✅ halfhourly_avg_v2 sauvegardé !")

In [2]:
from pyspark.sql import functions as F

storage_account = "energybigdatastorage"

# Lire les données sauvegardées
df_verification = spark.read.format("delta").load(
    f"abfss://processed@{storage_account}.dfs.core.windows.net/halfhourly_v2/")

print(f"Nombre de lignes : {df_verification.count()}")
print("\n=== SCHEMA ===")
df_verification.printSchema()
print("\n=== APERÇU ===")
df_verification.show(5)
print("\n=== VALEURS NULLES ===")
df_verification.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_verification.columns]).show()
print("\n=== OUTLIERS ===")
df_verification.groupBy("is_outlier").count().show()
print("\n=== IS_WEEKEND ===")
df_verification.groupBy("is_weekend").count().show()
print("\n=== QUARTER ===")
df_verification.groupBy("quarter").count().orderBy("quarter").show()